In [1]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix


In [6]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 50
EPOCHS_FINE = 50
LR_HEAD = 1e-4
LR_FINE = 1e-5

TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\train"
VAL_DIR   = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\val"
TEST_DIR  = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\test"

HEAD_HISTORY_JSON = "mobilenetv2_head_history_LWDCD.json"
FINE_HISTORY_JSON = "mobilenetv2_finetune_history_LWDCD.json"
HEAD_MODEL_PATH = "MobileNetV2_head_LWDCD.keras"
FINAL_MODEL_PATH = "MobileNetV2_finetuned_LWDCD.keras"
BEST_MODEL_PATH = "MobileNetV2_best_LWDCD.keras"

In [3]:
tf.random.set_seed(42)
np.random.seed(42)

In [4]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_generator.num_classes

Found 2351 images belonging to 3 classes.
Found 506 images belonging to 3 classes.
Found 505 images belonging to 3 classes.


In [5]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,591,811 (9.89 MB)

 Trainable params: 331,267 (1.26 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [7]:
model.compile(
    optimizer=Adam(learning_rate=LR_HEAD),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5389 - loss: 1.2079
Epoch 1: val_accuracy improved from -inf to 0.80435, saving model to MobileNetV2_best_LWDCD.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 136s 2s/step - accuracy: 0.5401 - loss: 1.2049 - val_accuracy: 0.8043 - val_loss: 0.5034 - learning_rate: 1.0000e-04
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7563 - loss: 0.6612
Epoch 2: val_accuracy improved from 0.80435 to 0.84980, saving model to MobileNetV2_best_LWDCD.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 125s 2s/step - accuracy: 0.7564 - loss: 0.6610 - val_accuracy: 0.8498 - val_loss: 0.3896 - learning_rate: 1.0000e-04
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8052 - loss: 0.5151
Epoch 3: val_accuracy improved from 0.84980 to 0.86957, saving model to MobileNetV2_best_LWDCD.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.8053 - loss: 0.5152 - val_accuracy: 0.8696 - val_loss: 0.3385 - learning_rate: 1.0000e-04
Epoch 4/50
74/74 ━━

ResourceExhaustedError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
MemoryError: 
Traceback (most recent call last):

  File "C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py", line 248, in _finite_generator
    yield self._standard

In [2]:
# Save head history
history_head_dict = {
    key: [float(x) for x in values]
    for key, values in history_head.history.items()
}

with open(HEAD_HISTORY_JSON, "w") as f:
    json.dump(history_head_dict, f, indent=4)

print(f"\nHead training history saved to: {HEAD_HISTORY_JSON}")

# Save model after head training
model.save(HEAD_MODEL_PATH)
print(f"Head-trained model saved to: {HEAD_MODEL_PATH}")


NameError: name 'history_head' is not defined

In [ ]:
base_model.trainable = True

# Freeze lower layers, fine-tune upper layers
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=LR_FINE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

In [ ]:
# Save fine-tuning history
history_fine_dict = {
    key: [float(x) for x in values]
    for key, values in history_fine.history.items()
}

with open(FINE_HISTORY_JSON, "w") as f:
    json.dump(history_fine_dict, f, indent=4)

print(f"\nFine-tuning history saved to: {FINE_HISTORY_JSON}")

# Save final model
model.save(FINAL_MODEL_PATH)
print(f"Final fine-tuned model saved to: {FINAL_MODEL_PATH}")


In [ ]:
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# =========================================================
# CLASSIFICATION REPORT + CONFUSION MATRIX
# =========================================================
pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))